[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/integrations/evaluating_traces_cleanlabTLM.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# 使用 Cleanlab 自动找出 LLM 评估中的不良回答

本指南介绍如何使用 Cleanlab 的 Trustworthy Language Models（TLM）评估 Phoenix 捕获的 LLM 回答。

TLM 会指出哪些模型回答不可信，从而提升 LLM 应用的可靠性。

本指南需要 Cleanlab TLM API 密钥。如果尚未获取，可在[此处](https://tlm.cleanlab.ai/)申请免费试用。

## 安装依赖并设置环境变量

In [ ]:
%%bash
pip install -q "arize-phoenix>=4.29.0"
pip install -q openai cleanlab_tlm openinference-instrumentation-openai

In [ ]:
import json
import os
from getpass import getpass

import dotenv

dotenv.load_dotenv()

In [ ]:
if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")

os.environ["OPENAI_API_KEY"] = openai_api_key

In [ ]:
# 在[这里](https://tlm.cleanlab.ai/) 注册 Cleanlab TLM 免费试用并获取 API 密钥。
if not (cleanlab_tlm_api_key := os.getenv("CLEANLAB_TLM_API_KEY")):
    cleanlab_tlm_api_key = getpass("🔑 请输入 Cleanlab TLM API 密钥： ")

os.environ["CLEANLAB_TLM_API_KEY"] = cleanlab_tlm_api_key

## 连接 Phoenix

本示例使用 Phoenix 作为追踪数据目的地，也可以添加其他需要的导出器。

如需设置 Phoenix API 密钥，请前往[此处](https://app.phoenix.arize.com/)。

以下代码连接到 Phoenix Cloud 实例，也可以连接[自托管 Phoenix 实例](https://arize.com/docs/phoenix/deployment)。

In [ ]:
# 添加用于追踪的 Phoenix API 密钥
if not (PHOENIX_API_KEY := os.getenv("PHOENIX_CLIENT_HEADERS")):
    PHOENIX_API_KEY = getpass("🔑 请输入 Phoenix API 密钥： ")
os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={PHOENIX_API_KEY}"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com"

配置 Phoenix 后，将该实例注册到 OpenTelemetry，即可在此收集应用追踪。

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="evaluating_traces_TLM")

## 准备追踪数据集

为了让本指南可以完整运行，先生成一些追踪并记录到 Phoenix。通常你已经在 Phoenix 中捕获了追踪，可以直接跳到“从 Phoenix 下载追踪数据集”。

In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

In [ ]:
from openai import OpenAI

# 初始化 OpenAI 客户端
client = OpenAI()


# 生成答案的函数
def generate_answers(trivia_question):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "你是知识问答大师。"},
            {"role": "user", "content": trivia_question},
        ],
    )
    answer = response.choices[0].message.content
    return answer


trivia_questions = [
    "What is the 3rd month of the year in alphabetical order?",
    "What is the capital of France?",
    "How many seconds are in 100 years?",
    "Alice, Bob, and Charlie went to a café. Alice paid twice as much as Bob, and Bob paid three times as much as Charlie. If the total bill was $72, how much did each person pay?",
    "When was the Declaration of Independence signed?",
]

# 生成答案
answers = []
for i in range(len(trivia_questions)):
    answer = generate_answers(trivia_questions[i])
    answers.append(answer)
    print(f"问题 {i + 1}: {trivia_questions[i]}")
    print(f"答案 {i + 1}:\n{answer}\n")

print(f"已生成 {len(answers)} 个答案，并在 Phoenix 中追踪。")

## 从 Phoenix 下载追踪数据集

In [ ]:
from phoenix.client import AsyncClient

px_client = AsyncClient()
spans_df = await px_client.spans.get_spans_dataframe(project_name="evaluating_traces_TLM")
spans_df.head()

## 使用 TLM 生成评估

获得追踪数据集后，使用 TLM 评估每条追踪。目标是为追踪中的每组提示词与回答生成可信度分数和解释。

In [ ]:
from cleanlab_tlm import TLM

tlm = TLM(options={"log": ["explanation"]})

首先从各条追踪中提取提示词和回答。`TLM.get_trustworthiness_score()` 接收提示词和回答列表，返回可信度分数及解释。

**重要：** 必须包含 LLM 生成原始回答时接收的所有系统提示词、上下文及其他信息。构造 `get_trustworthiness_score()` 的提示词输入时，应尽可能与原始提示词保持一致。

In [ ]:
# 创建包含输入和输出列的新 DataFrame
eval_df = spans_df[["context.span_id", "attributes.input.value", "attributes.output.value"]].copy()
eval_df.set_index("context.span_id", inplace=True)


# 合并追踪中的系统与用户提示词
def get_prompt(input_value):
    if isinstance(input_value, str):
        input_value = json.loads(input_value)
    system_prompt = input_value["messages"][0]["content"]
    user_prompt = input_value["messages"][1]["content"]
    return system_prompt + "\n" + user_prompt


# 从追踪获取回复
def get_response(output_value):
    if isinstance(output_value, str):
        output_value = json.loads(output_value)
    return output_value["choices"][0]["message"]["content"]


# 创建提示词及对应回复的列表
prompts = [get_prompt(input_value) for input_value in eval_df["attributes.input.value"]]
responses = [get_response(output_value) for output_value in eval_df["attributes.output.value"]]

eval_df["prompt"] = prompts
eval_df["response"] = responses

获得全部提示词和回答后，使用 TLM 评估每组配对。

In [ ]:
# 使用 TLM 评估每个提示词与回复对
evaluations = tlm.get_trustworthiness_score(prompts, responses)

# 从评估中提取可信度分数与说明
trust_scores = [entry["trustworthiness_score"] for entry in evaluations]
explanations = [entry["log"]["explanation"] for entry in evaluations]

# 将可信度分数与说明加入 DataFrame
eval_df["score"] = trust_scores
eval_df["explanation"] = explanations

# 显示新 DataFrame
eval_df.head()

现在的 DataFrame 新增了以下列：
- `prompt`：追踪中合并后的系统提示词和用户提示词
- `response`：追踪中的 LLM 回答
- `score`：TLM 可信度分数
- `explanation`：TLM 给出的解释

按 `score` 列排序，快速找出不可信的 LLM 回答。

In [ ]:
sorted_df = eval_df.sort_values(by="score", ascending=True).head()
sorted_df

In [ ]:
# 查看可信度最低的追踪。
print("提示词： ", sorted_df.iloc[0]["prompt"], "\n")
print("OpenAI 回复： ", sorted_df.iloc[0]["response"], "\n")
print("TLM 可信度分数： ", sorted_df.iloc[0]["score"], "\n")
print("TLM 说明： ", sorted_df.iloc[0]["explanation"])

#### TLM 成功识别出了多条包含 OpenAI 错误回答的追踪。

将 `score` 和 `explanation` 列上传到 Phoenix。

## 将评估上传到 Phoenix

我们的 evals_df 包含 span_id 列和评估结果列。span_id 将评估关联到 Phoenix 中正确的追踪。Phoenix 还会自动查找名为“score”和“evaluation”的列，并在界面中显示。

In [ ]:
eval_df["score"] = eval_df["score"].astype(float)
eval_df["explanation"] = eval_df["explanation"].astype(str)

In [ ]:
await px_client.spans.log_span_annotations_dataframe(
    dataframe=eval_df,
    annotation_name="Trustworthiness",
    annotator_kind="LLM",
)

现在应能在 Phoenix 界面中看到评估！

接下来可以继续收集和评估追踪。